# Model Context Protocol (MCP) â€” Practice Notebook

**Hexaview Technologies | AI Engineering Series**

---

## Overview
This notebook accompanies the *MCP: Model Context Protocol* tutorial. You will build MCP servers in Python using the **FastMCP** SDK, progressing from simple tools to a complete file explorer server.

**Estimated time:** 4â€“5 hours  
**Prerequisites:** Python 3.10+, basic Python knowledge, no prior MCP experience  
**Reference:** `docs/MCP_Tutorial_Hexaview.pdf`

### How to use this notebook
- Each exercise has **skeleton code** with `# TODO` comments â€” fill them in
- Expand the **Hint** sections if you get stuck
- Compare your output against the **Expected Output** sections
- Exercises build on each other â€” complete them in order

---
## Setup & Installation (~15 min)

In [ ]:
# Install the MCP SDK
!pip install mcp pyyaml --quiet

In [ ]:
# Verify installation
from mcp.server.fastmcp import FastMCP
print("FastMCP imported successfully!")
print(f"Ready to build MCP servers.")

---
## Exercise 1: Conceptual Understanding (~20 min)

Before writing code, let's make sure you understand the core MCP concepts.

### 1a. Fill in the blanks
Replace each `"___"` with the correct answer.

In [ ]:
# EXERCISE 1a: Fill in the blanks about MCP architecture

# The three actors in MCP are:
actor_1 = "___"  # TODO: The application the user runs (e.g., Claude Desktop, VS Code, Cursor, your custom chat UI)
actor_2 = "___"  # TODO: Lives inside the Host, manages connections to servers
actor_3 = "___"  # TODO: What YOU build â€” exposes capabilities over JSON-RPC

# The three core primitives are:
primitive_actions = "___"    # TODO: Functions the AI can CALL (send email, run query)
primitive_readonly = "___"   # TODO: Data the AI can READ (handbook, config file)
primitive_templates = "___"  # TODO: Reusable prompt TEMPLATES (summarise ticket)

# Check your answers
answers = {
    "actors": [actor_1, actor_2, actor_3],
    "primitives": [primitive_actions, primitive_readonly, primitive_templates]
}
print("Your answers:", answers)

<details>
<summary><strong>Hint â€” Exercise 1a</strong></summary>

**Actors:** Host, Client, Server  
**Primitives:** Tools (DO something), Resources (READ something), Prompts (SAY something)

</details>

**Expected Output:**
```
Your answers: {'actors': ['Host', 'Client', 'Server'], 'primitives': ['Tools', 'Resources', 'Prompts']}
```

### 1b. Match the primitive to the use case
For each use case, write `"Tool"`, `"Resource"`, or `"Prompt"`.

In [ ]:
# EXERCISE 1b: Classify each use case

use_cases = {
    "Send an email to a customer":           "___",  # TODO
    "Read the company handbook":              "___",  # TODO
    "Standardise how AI summarises tickets":  "___",  # TODO
    "Query a database":                       "___",  # TODO
    "Fetch a config file":                    "___",  # TODO
    "Template for writing release notes":     "___",  # TODO
}

for case, answer in use_cases.items():
    print(f"  {answer:>10}  |  {case}")

<details>
<summary><strong>Hint â€” Exercise 1b</strong></summary>

- **Tool** = DO something (create, update, delete, send, run) â€” has side effects  
- **Resource** = READ something (fetch, view, retrieve) â€” read-only  
- **Prompt** = SAY something (instruct, template, standardise) â€” reusable template

</details>

**Expected Output:**
```
      Tool  |  Send an email to a customer
  Resource  |  Read the company handbook
    Prompt  |  Standardise how AI summarises tickets
      Tool  |  Query a database
  Resource  |  Fetch a config file
    Prompt  |  Template for writing release notes
```

### 1c. MCP vs Function Calling â€” Key difference
In your own words, explain the key difference.

In [ ]:
# EXERCISE 1c: Explain in one sentence
# TODO: Replace the string below with your answer

mcp_vs_function_calling = "___"

print(f"My answer: {mcp_vs_function_calling}")

<details>
<summary><strong>Hint â€” Exercise 1c</strong></summary>

Function calling is a feature **inside** an app (one integration, one app).  
MCP is a **reusable, interoperable component** â€” write once, use everywhere with any MCP-compatible client.

</details>

---
## Exercise 2: Your First MCP Server â€” Greeter (~45 min)

Build a FastMCP server with a single `greet` tool that returns a friendly greeting.

**Key concepts:**
- `FastMCP` is the high-level Python API â€” decorators instead of manual schema
- `@mcp.tool()` registers a function as an MCP tool
- Type hints become the input schema automatically
- The docstring becomes the tool description the AI reads

In [ ]:
# EXERCISE 2: Build a Greeter MCP Server

from mcp.server.fastmcp import FastMCP

# Step 1: Create the server instance
# TODO: Create a FastMCP server named "my-first-mcp"
mcp_greeter = None  # Replace with: FastMCP("...")


# Step 2: Register a tool using the @mcp.tool() decorator
# TODO: Add the correct decorator and implement the function
# The function should:
#   - Be named 'greet'
#   - Accept a 'name' parameter (str)
#   - Return a str
#   - Have a docstring: "Returns a friendly greeting for a given name."
#   - Return: f"Hello, {name}! This came from your Python MCP server!"

# YOUR CODE HERE:



print("Server created:", mcp_greeter.name if mcp_greeter else "NOT CREATED")

<details>
<summary><strong>Hint â€” Exercise 2</strong></summary>

```python
mcp_greeter = FastMCP("my-first-mcp")

@mcp_greeter.tool()
def greet(name: str) -> str:
    """Returns a friendly greeting for a given name."""
    return f"Hello, {name}! This came from your Python MCP server!"
```

Note: We use `@mcp_greeter.tool()` (our variable name), not `@mcp.tool()`.

</details>

In [ ]:
# Test your greeter tool by calling it directly
# (In production, the MCP client calls this â€” here we test it manually)

result = greet("Alice")
print("Tool result:", result)

# Verify
assert result == "Hello, Alice! This came from your Python MCP server!", "Greeting doesn't match expected output!"
print("\nAll assertions passed!")

**Expected Output:**
```
Tool result: Hello, Alice! This came from your Python MCP server!

All assertions passed!
```

In [ ]:
# EXERCISE 2b: Inspect the auto-generated tool schema
# FastMCP builds a JSON schema from your type hints and docstring.
# Let's see what it looks like.

import asyncio
from mcp.types import ListToolsRequest

async def inspect_tools(server):
    """List all tools registered on the server."""
    tools = await server.list_tools()
    for tool in tools:
        print(f"Tool: {tool.name}")
        print(f"  Description: {tool.description}")
        print(f"  Input Schema: {tool.inputSchema}")
        print()

await inspect_tools(mcp_greeter)

**Expected Output (approximate):**
```
Tool: greet
  Description: Returns a friendly greeting for a given name.
  Input Schema: {'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': '...', 'type': 'object'}
```

Notice how FastMCP automatically converted your `name: str` parameter into a JSON Schema!

---
## Exercise 3: Adding Resources (~30 min)

Resources are **read-only data** the AI can access. Think of them as files or documents the AI can request.

Add a resource to expose a "company handbook" that the AI can read.

In [ ]:
# EXERCISE 3: Add a Resource to the server

from mcp.server.fastmcp import FastMCP

mcp_with_resource = FastMCP("resource-demo")

# TODO: Register a resource using the @mcp_with_resource.resource() decorator
# Requirements:
#   - URI: "company://docs/handbook"
#   - Function name: get_handbook
#   - Returns a str
#   - Docstring: "Company Handbook"
#   - Return value: "Welcome to the company handbook. Our values are: Innovation, Integrity, Impact."

# YOUR CODE HERE:



print("Resource registered!")

<details>
<summary><strong>Hint â€” Exercise 3</strong></summary>

```python
@mcp_with_resource.resource("company://docs/handbook")
def get_handbook() -> str:
    """Company Handbook"""
    return "Welcome to the company handbook. Our values are: Innovation, Integrity, Impact."
```

Key difference from tools:
- `@mcp.resource(uri)` takes a URI string as argument
- Resources are read-only â€” they return data but don't change anything
- The URI is how the AI identifies and requests this resource

</details>

In [ ]:
# Test: Read the resource
result = get_handbook()
print("Resource content:", result)

assert "Innovation" in result, "Handbook should mention Innovation"
print("\nAssertion passed!")

**Expected Output:**
```
Resource content: Welcome to the company handbook. Our values are: Innovation, Integrity, Impact.

Assertion passed!
```

---
## Exercise 4: Adding Prompts (~30 min)

Prompts are **reusable prompt templates** that standardise how AI approaches a task. They're like pre-written instructions.

Create a prompt template for summarising support tickets.

In [ ]:
# EXERCISE 4: Add a Prompt template

from mcp.server.fastmcp import FastMCP

mcp_with_prompt = FastMCP("prompt-demo")

# TODO: Register a prompt using the @mcp_with_prompt.prompt() decorator
# Requirements:
#   - Function name: summarize_ticket
#   - Parameter: ticket_id (str)
#   - Returns a str
#   - Docstring: "Summarise a support ticket in 2 sentences."
#   - Return: f"Please summarise support ticket #{ticket_id} in exactly 2 sentences."

# YOUR CODE HERE:



print("Prompt registered!")

<details>
<summary><strong>Hint â€” Exercise 4</strong></summary>

```python
@mcp_with_prompt.prompt()
def summarize_ticket(ticket_id: str) -> str:
    """Summarise a support ticket in 2 sentences."""
    return f"Please summarise support ticket #{ticket_id} in exactly 2 sentences."
```

Prompts work like tools but with `@mcp.prompt()` â€” no URI needed (unlike resources).

</details>

In [ ]:
# Test the prompt
result = summarize_ticket("TICKET-4521")
print("Prompt output:", result)

assert "TICKET-4521" in result, "Prompt should include the ticket ID"
assert "2 sentences" in result, "Prompt should mention 2 sentences"
print("\nAll assertions passed!")

**Expected Output:**
```
Prompt output: Please summarise support ticket #TICKET-4521 in exactly 2 sentences.

All assertions passed!
```

---
## Exercise 5: File Explorer MCP Server (~60 min)

This is the main challenge. You will build a **real-world MCP server** that lets an AI:
1. **List directories** â€” see what files exist
2. **Read files** â€” view file contents

**Critical security concept:** The `safe_path()` function ensures the AI can ONLY access files within an allowed directory. Without this, an AI could read `/etc/passwd` or your SSH keys!

In [ ]:
# EXERCISE 5: File Explorer MCP Server

import os
from pathlib import Path
from mcp.server.fastmcp import FastMCP

# The safe root: our server can ONLY read files inside this folder
ALLOWED_ROOT = Path(os.path.join(os.path.dirname(os.path.abspath("__file__")), "test_data", "sample_project")).resolve()
print(f"ALLOWED_ROOT: {ALLOWED_ROOT}")

mcp_explorer = FastMCP("file-explorer")


# --- STEP 1: Implement the safe_path() security function ---
# TODO: Complete this function
def safe_path(requested: str) -> Path:
    """
    Resolve the requested path and verify it stays within ALLOWED_ROOT.
    Raise ValueError if the path is outside the allowed directory.
    """
    # TODO: 1. Resolve the requested path to an absolute path using Path(requested).resolve()
    # TODO: 2. Check if the resolved path starts with ALLOWED_ROOT (use str() and startswith())
    # TODO: 3. If NOT within ALLOWED_ROOT, raise ValueError("Access denied: path is outside the allowed directory")
    # TODO: 4. Return the resolved path
    
    raise NotImplementedError("Implement the safe_path function")


# --- STEP 2: Implement the list_directory tool ---
# TODO: Add the @mcp_explorer.tool() decorator and implement
def list_directory(dir_path: str) -> str:
    """
    List all files and folders inside a given directory.
    Provide the absolute path of the directory to list.
    """
    # TODO: 1. Use safe_path() to validate the path
    # TODO: 2. Get all entries using target.iterdir()
    # TODO: 3. For each entry, format as "[DIR] name" or "[FILE] name"
    # TODO: 4. Return the joined lines (or "(empty directory)" if none)
    # Wrap everything in try/except to return errors as text
    
    raise NotImplementedError("Implement the list_directory tool")


# --- STEP 3: Implement the read_file tool ---
# TODO: Add the @mcp_explorer.tool() decorator and implement
def read_file(file_path: str) -> str:
    """
    Read and return the text content of a file.
    Provide the absolute path of the file to read.
    """
    # TODO: 1. Use safe_path() to validate the path
    # TODO: 2. Read and return the file content using target.read_text(encoding='utf-8')
    # Wrap in try/except to return errors as text
    
    raise NotImplementedError("Implement the read_file tool")


print("File Explorer server defined!")

<details>
<summary><strong>Hint â€” Exercise 5 (safe_path)</strong></summary>

```python
def safe_path(requested: str) -> Path:
    """Resolve path and verify it stays within ALLOWED_ROOT."""
    resolved = Path(requested).resolve()
    if not str(resolved).startswith(str(ALLOWED_ROOT)):
        raise ValueError(f"Access denied: path is outside the allowed directory")
    return resolved
```

</details>

<details>
<summary><strong>Hint â€” Exercise 5 (list_directory)</strong></summary>

```python
@mcp_explorer.tool()
def list_directory(dir_path: str) -> str:
    """List all files and folders inside a given directory.
    Provide the absolute path of the directory to list."""
    try:
        target = safe_path(dir_path)
        entries = list(target.iterdir())
        if not entries:
            return "(empty directory)"
        lines = []
        for entry in sorted(entries):
            prefix = "[DIR] " if entry.is_dir() else "[FILE]"
            lines.append(f"{prefix} {entry.name}")
        return "\n".join(lines)
    except Exception as e:
        return f"Error: {e}"
```

</details>

<details>
<summary><strong>Hint â€” Exercise 5 (read_file)</strong></summary>

```python
@mcp_explorer.tool()
def read_file(file_path: str) -> str:
    """Read and return the text content of a file.
    Provide the absolute path of the file to read."""
    try:
        target = safe_path(file_path)
        return target.read_text(encoding='utf-8')
    except Exception as e:
        return f"Error: {e}"
```

</details>

In [ ]:
# TEST 5a: Test safe_path with a valid path
valid = safe_path(str(ALLOWED_ROOT / "README.md"))
print(f"Valid path resolved: {valid}")

# TEST 5b: Test safe_path rejects paths outside ALLOWED_ROOT
try:
    safe_path("/etc/passwd")  # This should raise ValueError
    print("FAIL: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly blocked: {e}")

print("\nSecurity tests passed!")

In [ ]:
# TEST 5c: Test list_directory
print("=== Root directory ===")
print(list_directory(str(ALLOWED_ROOT)))

print("\n=== src/ directory ===")
print(list_directory(str(ALLOWED_ROOT / "src")))

**Expected Output (approximate):**
```
=== Root directory ===
[DIR]  docs
[FILE] README.md
[DIR]  src

=== src/ directory ===
[FILE] main.py
[FILE] utils.py
```

In [ ]:
# TEST 5d: Test read_file
print("=== README.md ===")
print(read_file(str(ALLOWED_ROOT / "README.md")))

print("\n=== src/main.py ===")
print(read_file(str(ALLOWED_ROOT / "src" / "main.py")))

**Expected Output:**
```
=== README.md ===
# Sample Project

This is a sample project used for the MCP File Explorer exercise.
It contains some example files to test your MCP server against.


=== src/main.py ===
"""Sample main module for testing the File Explorer MCP server."""

def hello():
    return "Hello from the sample project!"

if __name__ == "__main__":
    print(hello())
```

---
## Exercise 6: Multi-Tool Server â€” Task Manager (~45 min)

Now build a server **from scratch** with less scaffolding. Create a Task Manager with three tools:
- `add_task` â€” add a new task (takes a `title` string)
- `list_tasks` â€” show all tasks with their status
- `complete_task` â€” mark a task as done (takes a `task_id` int)

Use an in-memory list to store tasks.

In [ ]:
# EXERCISE 6: Task Manager MCP Server
# Build this from scratch â€” less scaffolding this time!

from mcp.server.fastmcp import FastMCP

mcp_tasks = FastMCP("task-manager")

# In-memory task storage
tasks = []  # Each task: {"id": int, "title": str, "done": bool}
next_id = 1

# TODO: Implement three tools:
#
# 1. add_task(title: str) -> str
#    - Add a task to the list with done=False
#    - Increment next_id
#    - Return confirmation with the task ID
#
# 2. list_tasks() -> str
#    - Return all tasks formatted as: "[x] #1: Buy milk" or "[ ] #2: Write code"
#    - Return "No tasks yet." if empty
#
# 3. complete_task(task_id: int) -> str
#    - Find the task by ID and set done=True
#    - Return confirmation or "Task not found" error

# YOUR CODE HERE:







print("Task Manager server defined!")

<details>
<summary><strong>Hint â€” Exercise 6</strong></summary>

```python
@mcp_tasks.tool()
def add_task(title: str) -> str:
    """Add a new task to the task list."""
    global next_id
    tasks.append({"id": next_id, "title": title, "done": False})
    task_id = next_id
    next_id += 1
    return f"Task #{task_id} added: {title}"

@mcp_tasks.tool()
def list_tasks() -> str:
    """List all tasks with their completion status."""
    if not tasks:
        return "No tasks yet."
    lines = []
    for t in tasks:
        status = "[x]" if t["done"] else "[ ]"
        lines.append(f"{status} #{t['id']}: {t['title']}")
    return "\n".join(lines)

@mcp_tasks.tool()
def complete_task(task_id: int) -> str:
    """Mark a task as completed by its ID."""
    for t in tasks:
        if t["id"] == task_id:
            t["done"] = True
            return f"Task #{task_id} completed: {t['title']}"
    return f"Task #{task_id} not found."
```

</details>

In [ ]:
# Test the Task Manager
print("--- Adding tasks ---")
print(add_task("Buy groceries"))
print(add_task("Write MCP server"))
print(add_task("Review pull request"))

print("\n--- All tasks ---")
print(list_tasks())

print("\n--- Completing task #2 ---")
print(complete_task(2))

print("\n--- Updated tasks ---")
print(list_tasks())

print("\n--- Try completing non-existent task ---")
print(complete_task(99))

**Expected Output:**
```
--- Adding tasks ---
Task #1 added: Buy groceries
Task #2 added: Write MCP server
Task #3 added: Review pull request

--- All tasks ---
[ ] #1: Buy groceries
[ ] #2: Write MCP server
[ ] #3: Review pull request

--- Completing task #2 ---
Task #2 completed: Write MCP server

--- Updated tasks ---
[ ] #1: Buy groceries
[x] #2: Write MCP server
[ ] #3: Review pull request

--- Try completing non-existent task ---
Task #99 not found.
```

---
## Exercise 7: HTTP + SSE Transport (~30 min)

So far, all servers use **stdio** transport (local, single-user). For remote/multi-user deployment, MCP supports **HTTP + SSE**.

In this exercise, you'll write a server script that runs over HTTP instead of stdio.

In [ ]:
# EXERCISE 7: Write an HTTP+SSE server script
# We'll write this to a .py file since HTTP servers need to run as a process.

server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-remote-mcp")

@mcp.tool()
def greet(name: str) -> str:
    """Returns a greeting."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    # TODO: Change the mcp.run() call to use HTTP+SSE transport
    # Instead of the default stdio, use:
    #   transport="sse", host="0.0.0.0", port=3000
    
    mcp.run()  # FIX THIS LINE
'''

print("Current server code (needs fixing):")
print(server_code)

In [ ]:
# TODO: Write the corrected server code below
# Replace mcp.run() with the correct HTTP+SSE transport call

corrected_server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-remote-mcp")

@mcp.tool()
def greet(name: str) -> str:
    """Returns a greeting."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    # YOUR FIX HERE:
    pass  # Replace this line
'''

# Save to file
with open("test_data/sse_server.py", "w") as f:
    f.write(corrected_server_code)

print("Server script saved to test_data/sse_server.py")
print("\nTo run it: python test_data/sse_server.py")
print("It should print: 'MCP HTTP server listening on http://localhost:3000'")

<details>
<summary><strong>Hint â€” Exercise 7</strong></summary>

Replace the `mcp.run()` line with:

```python
mcp.run(transport="sse", host="0.0.0.0", port=3000)
```

That's it! FastMCP handles all the HTTP + SSE plumbing internally.  
Compare this to the TypeScript version which requires Express, SSEServerTransport, and manual route setup.

</details>

In [ ]:
# Verify your corrected code contains the right transport call
assert 'transport="sse"' in corrected_server_code or "transport='sse'" in corrected_server_code, \
    "Your code should contain transport='sse'"
assert 'port=3000' in corrected_server_code or 'port = 3000' in corrected_server_code, \
    "Your code should specify port=3000"

print("Verification passed! Your server is configured for HTTP+SSE transport.")
print("\nKey differences from stdio:")
print("  stdio  -> local child process, no port, no auth needed")
print("  HTTP+SSE -> remote web server, runs on port 3000, needs auth in production")

---
## Exercise 8: Connecting an LLM to Your MCP Server (BONUS — Optional) (~45 min)

**Requires:** OpenAI API key in your `.env` file, `pip install openai`

In this exercise you'll wire an LLM (via OpenAI's API) to discover and call your MCP tools. This demonstrates the full loop:

```
User Question -> LLM decides to call tool -> MCP Server executes -> Result back to LLM -> Final answer
```

In [ ]:
# EXERCISE 8 (BONUS): LLM + MCP Integration
# Skip this exercise if you don't have an OpenAI API key.

import os
from pathlib import Path

# Load API key from .env file
# SECURITY: Never hardcode API keys in notebooks!
env_path = Path("..") / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if not OPENAI_API_KEY or OPENAI_API_KEY == "your-api-key-here":
    print("No OPENAI_API_KEY found.")
    print("  1. Copy .env.example to .env:  cp .env.example .env")
    print("  2. Edit .env and paste your OpenAI API key")
    print("  3. Re-run this cell")
    print("
Skipping this exercise is fine — it's a bonus!")
    OPENAI_API_KEY = ""
else:
    print("OpenAI API key loaded from .env. Let's connect an LLM to our MCP server!")

In [ ]:
# EXERCISE 8a: Convert MCP tool schema to OpenAI function format
#
# MCP tools have: name, description, inputSchema
# OpenAI expects: {"type": "function", "function": {"name": ..., "description": ..., "parameters": ...}}

# Here's our MCP tool definition (from the greeter):
mcp_tool = {
    "name": "greet",
    "description": "Returns a friendly greeting for a given name.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The name of the person to greet"}
        },
        "required": ["name"]
    }
}

# TODO: Write a function that converts MCP tool format to OpenAI function format
def mcp_to_openai_tool(mcp_tool: dict) -> dict:
    """
    Convert an MCP tool definition to OpenAI function calling format.
    
    Expected output format:
    {
        "type": "function",
        "function": {
            "name": "...",
            "description": "...",
            "parameters": { ... }  # same as inputSchema
        }
    }
    """
    # YOUR CODE HERE:
    raise NotImplementedError("Convert MCP tool to OpenAI format")


# Test the conversion
openai_tool = mcp_to_openai_tool(mcp_tool)
print("OpenAI tool format:")
import json
print(json.dumps(openai_tool, indent=2))

<details>
<summary><strong>Hint â€” Exercise 8a</strong></summary>

```python
def mcp_to_openai_tool(mcp_tool: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": mcp_tool["name"],
            "description": mcp_tool["description"],
            "parameters": mcp_tool["inputSchema"]
        }
    }
```

</details>

In [ ]:
# EXERCISE 8b: Build the full LLM + MCP loop
# This simulates what an MCP Host (like Claude Desktop, Cursor, or any MCP-compatible host) does.

import json

# Our local "MCP server" â€” just the greet function from Exercise 2
def execute_mcp_tool(tool_name: str, arguments: dict) -> str:
    """Simulate MCP tool execution."""
    if tool_name == "greet":
        name = arguments.get("name", "World")
        return f"Hello, {name}! This came from your Python MCP server!"
    return f"Unknown tool: {tool_name}"


# TODO: Complete the agent loop
# This function should:
# 1. Send the user message to the LLM with tool definitions
# 2. If the LLM calls a tool, execute it via execute_mcp_tool()
# 3. Send the tool result back to the LLM
# 4. Return the final LLM response

async def agent_with_mcp(user_message: str) -> str:
    """
    Send a message to the LLM, let it use MCP tools, return the final answer.
    """
    if not OPENAI_API_KEY:
        return "Skipped â€” no API key"
    
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    tools = [mcp_to_openai_tool(mcp_tool)]
    messages = [{"role": "user", "content": user_message}]
    
    # TODO: Step 1 â€” Call the LLM with tools
    # response = client.chat.completions.create(
    #     model="gpt-4o-mini",
    #     messages=messages,
    #     tools=tools
    # )
    
    # TODO: Step 2 â€” Check if the LLM wants to call a tool
    # choice = response.choices[0]
    # if choice.finish_reason == "tool_calls":
    #     for tool_call in choice.message.tool_calls:
    #         args = json.loads(tool_call.function.arguments)
    #         result = execute_mcp_tool(tool_call.function.name, args)
    #         # Append assistant message and tool result to messages
    #         # Then call the LLM again to get the final answer
    
    # TODO: Step 3 â€” Return the final response text
    
    raise NotImplementedError("Complete the agent loop")


# Test it!
result = await agent_with_mcp("Please greet Alice for me.")
print("Agent response:", result)

<details>
<summary><strong>Hint â€” Exercise 8b (Full Solution)</strong></summary>

```python
async def agent_with_mcp(user_message: str) -> str:
    if not OPENAI_API_KEY:
        return "Skipped â€” no API key"
    
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    tools = [mcp_to_openai_tool(mcp_tool)]
    messages = [{"role": "user", "content": user_message}]
    
    # Step 1: Call LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )
    choice = response.choices[0]
    
    # Step 2: Handle tool calls
    if choice.finish_reason == "tool_calls":
        messages.append(choice.message)
        for tool_call in choice.message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = execute_mcp_tool(tool_call.function.name, args)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        # Call LLM again with tool results
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        choice = response.choices[0]
    
    # Step 3: Return final answer
    return choice.message.content
```

</details>

**Expected Output (approximate â€” LLM wording will vary):**
```
Agent response: Hello, Alice! This came from your Python MCP server!
```

Or the LLM might rephrase it like:
```
Agent response: I greeted Alice for you! The response was: "Hello, Alice! This came from your Python MCP server!"
```

---
## Summary & Key Takeaways

### What you built:
1. A **Greeter** MCP server with a tool, resource, and prompt
2. A **File Explorer** with path security validation
3. A **Task Manager** with CRUD operations
4. An **HTTP+SSE** transport configuration
5. (Bonus) An **LLM agent** that calls MCP tools

### Key rules to remember:
- **Never use `print()`** with stdio transport â€” use `logging` to stderr instead
- **Always validate paths** with `safe_path()` â€” never let AI access unrestricted files
- **Type hints + docstrings** = automatic schema in FastMCP
- **Return errors as text**, not exceptions â€” so the AI can read and explain them
- Use **stdio** for local dev, **HTTP+SSE** for remote deployment

### What to explore next:
- Connect your server to an MCP-compatible host (Claude Desktop, Cursor, VS Code, etc.)
- Build a server that talks to a real API (GitHub, JIRA, database)
- Use the MCP Inspector for debugging: `npx @modelcontextprotocol/inspector python3 server.py`
- Combine MCP servers with Skills (see Notebook 2)